In [32]:
from udemy_purchase_scraper.utils import loadPurchases, loadOwned

In [33]:
owned = loadOwned()
purchases = loadPurchases()

In [35]:
from udemy_purchase_scraper.utils import uniqueCourses
owned = uniqueCourses(owned)

In [36]:
# Not Needed In Final
for ind, course in enumerate(owned):
        owned[ind].course_link = f"https://www.udemy.com{course.course_link}"

In [ ]:
from udemy_purchase_scraper.owned.scraper import getOwnedCurrentURLs
owned = getOwnedCurrentURLs(owned)

In [ ]:
# Don't Run Above

In [22]:
owned_courses_dict = {}
for course in owned:
    owned_courses_dict[course.course_page] = course

In [23]:
purchase_courses_dict = {}
for purchase in purchases:
    for course in purchase.courses:
        purchase_courses_dict[course.course_link] = None

In [24]:
missing_owned_courses = set(owned_courses_dict.keys()).difference(set(purchase_courses_dict.keys()))
free_courses = [owned_courses_dict[course] for course in missing_owned_courses]
del purchase_courses_dict, missing_owned_courses

In [25]:
original_lengths = [len(purchase.courses) for purchase in purchases]

for purchase in purchases:
    purchase.courses = [course for course in purchase.courses if course.course_link in owned_courses_dict]

new_lengths = [len(purchase.courses) for purchase in purchases]

In [26]:
for ind, purchase in enumerate(purchases):
    if original_lengths[ind] != new_lengths[ind] and new_lengths[ind] != 0:
        purchases[ind].purchase_cost = sum([course.course_cost for course in purchases[ind].courses])
del original_lengths, new_lengths

purchases = [purchase for purchase in purchases if len(purchase.courses) != 0]

In [27]:
from udemy_purchase_scraper.owned.classes import UdemyCoursePurchaseDetails
course_prices = {}
for purchase in purchases:
    for course in purchase.courses:
        course_prices[course.course_link] = UdemyCoursePurchaseDetails(
            course_purchase=purchase.purchase_number, 
            course_date=purchase.purchase_date, 
            course_cost=course.course_cost)

In [28]:
for crs_ind, course in enumerate(owned):
    if course.course_page in course_prices:
        owned[crs_ind].purchase = course_prices[course.course_page]
del course_prices

In [29]:
for p_ind, purchase in enumerate(purchases):
    for c_ind, course in enumerate(purchase.courses):
        purchases[p_ind].courses[c_ind] = owned_courses_dict[course.course_link]
del owned_courses_dict